In [3]:
from tqdm.autonotebook import tqdm, trange

In [6]:
pip install sentence-transformers

     |████████████████████████████████| 255 kB 2.2 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re
import nltk
from nltk.tokenize import sent_tokenize

In [4]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/raham/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
document_sentences_df = pd.read_csv('all_segmented_sentences_from_articles.csv')

# Rename columns to match our code (adjust these column names to match your CSV)
document_sentences_df = document_sentences_df.rename(columns={
    'article_id': 'document_id',           # or whatever your article ID column is called
    'sentence_number': 'sentence_number',  # or whatever your sentence number column is called
    'sentence': 'sentence_text'       # or whatever your sentence text column is called
})

# Create unique sentence IDs
document_sentences_df['sentence_id'] = (
    document_sentences_df['document_id'].astype(str) + '_' + 
    document_sentences_df['sentence_number'].astype(str)
)

print(f"Loaded {len(document_sentences_df)} sentences from {document_sentences_df['document_id'].nunique()} documents")

Loaded 13660 sentences from 118 documents


In [6]:
labeled_sentences_df = pd.read_csv('annotated_corpus_for_dataverse.csv')

# Show all unique labels in your data
print(f"Unique labels in your data: {sorted(labeled_sentences_df['relevance_label'].unique())}")

# Filter for relevance_label 1 and 2 only
relevant_sentences_df = labeled_sentences_df[labeled_sentences_df['relevance_label'].isin([1, 2])]
relevant_sentences = relevant_sentences_df['text_segment'].tolist()

print(f"Total labeled sentences: {len(labeled_sentences_df)}")
print(f"Sentences with label 1: {len(labeled_sentences_df[labeled_sentences_df['relevance_label'] == 1])}")
print(f"Sentences with label 2: {len(labeled_sentences_df[labeled_sentences_df['relevance_label'] == 2])}")
print(f"Total relevant sentences (labels 1,2) to use as queries: {len(relevant_sentences)}")

# Show sample of your relevant sentences
print(f"\nSample relevant sentences:")
for i, sentence in enumerate(relevant_sentences[:3]):
    label = relevant_sentences_df.iloc[i]['relevance_label']
    print(f"Label {label}: {sentence[:100]}...")

Unique labels in your data: [0, 1, 2]
Total labeled sentences: 803
Sentences with label 1: 166
Sentences with label 2: 67
Total relevant sentences (labels 1,2) to use as queries: 233

Sample relevant sentences:
Label 2: #text': 'After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was ...
Label 1: #text': 'Pastoral production has often existed in some sort of symbiotic interaction with the [agric...
Label 2: #text': 'The forests of West and Central Africa probably originally covered a combined area of about...


In [7]:
def cross_encoder_similarity_search(relevant_sentences, document_sentences_df, 
                                   threshold=0.5, model_name='cross-encoder/ms-marco-MiniLM-L-6-v2',
                                   batch_size=32, max_doc_sentences=None):

    
    print(f"Loading Cross-Encoder model: {model_name}")
    cross_encoder = CrossEncoder(model_name)
    
    # Limit document sentences if specified (for testing)
    if max_doc_sentences:
        doc_df = document_sentences_df.head(max_doc_sentences).copy()
        print(f"Processing first {max_doc_sentences} document sentences for testing")
    else:
        doc_df = document_sentences_df.copy()
    
    print(f"Cross-encoder setup:")
    print(f"  Queries: {len(relevant_sentences)}")
    print(f"  Document sentences: {len(doc_df)}")
    print(f"  Total comparisons: {len(relevant_sentences) * len(doc_df):,}")
    print(f"  Batch size: {batch_size}")
    print(f"  Threshold: {threshold}")
    
    all_results = []
    start_time = time.time()
    
    # Process each query
    for query_idx, query_text in enumerate(relevant_sentences):
        print(f"\nProcessing Query {query_idx + 1}/{len(relevant_sentences)}")
        print(f"Query: '{query_text[:100]}...'")
        
        # Create sentence pairs for this query
        sentence_pairs = []
        doc_indices = []
        
        for doc_idx, row in doc_df.iterrows():
            sentence_pairs.append([query_text, row['sentence_text']])
            doc_indices.append(doc_idx)
        
        # Process in batches
        query_results = []
        
        for i in tqdm(range(0, len(sentence_pairs), batch_size), 
                     desc=f"Query {query_idx + 1} batches"):
            
            batch_pairs = sentence_pairs[i:i + batch_size]
            batch_indices = doc_indices[i:i + batch_size]
            
            # Get similarity scores for this batch
            batch_scores = cross_encoder.predict(batch_pairs)
            
            # Store results above threshold
            for j, score in enumerate(batch_scores):
                if score >= threshold:
                    doc_idx = batch_indices[j]
                    doc_row = doc_df.loc[doc_idx]
                    
                    result = {
                        'sentence_id': doc_row['sentence_id'],
                        'document_id': doc_row['document_id'],
                        'sentence_number': doc_row['sentence_number'],
                        'sentence_text': doc_row['sentence_text'],
                        'similarity_score': float(score),
                        'query_index': query_idx,
                        'query_text': query_text
                    }
                    query_results.append(result)
        
        print(f"  Found {len(query_results)} sentences above threshold {threshold}")
        all_results.extend(query_results)
    
    # Create final results DataFrame
    if all_results:
        results_df = pd.DataFrame(all_results)
        
        # Remove duplicates (same sentence matched by multiple queries)
        # Keep the one with highest similarity score
        results_df = results_df.sort_values('similarity_score', ascending=False)
        results_df = results_df.drop_duplicates(subset=['sentence_id'], keep='first')
        
        # Sort by similarity score
        results_df = results_df.sort_values('similarity_score', ascending=False)
        results_df = results_df.reset_index(drop=True)
        
        elapsed_time = time.time() - start_time
        print(f"\n" + "="*60)
        print(f"CROSS-ENCODER SEARCH COMPLETED")
        print(f"="*60)
        print(f"Total processing time: {elapsed_time/60:.2f} minutes")
        print(f"Total relevant sentences found: {len(results_df)}")
        print(f"Unique documents with relevant sentences: {results_df['document_id'].nunique()}")
        print(f"Average similarity score: {results_df['similarity_score'].mean():.3f}")
        print(f"Highest similarity score: {results_df['similarity_score'].max():.3f}")
        
    else:
        results_df = pd.DataFrame()
        print(f"\nNo sentences found above threshold {threshold}")
        print("Consider lowering the threshold or checking your queries")
    
    return results_df

In [8]:
def analyze_cross_encoder_results(results_df, relevant_sentences):
    """
    Detailed analysis of cross-encoder results
    """
    
    if len(results_df) == 0:
        print("No results to analyze")
        return
    
    print("\n" + "="*70)
    print("DETAILED CROSS-ENCODER ANALYSIS")
    print("="*70)
    
    # Query performance analysis
    print("\n📊 QUERY PERFORMANCE:")
    query_performance = results_df.groupby('query_index').agg({
        'similarity_score': ['count', 'mean', 'max'],
        'document_id': 'nunique'
    }).round(3)
    
    query_performance.columns = ['Matches', 'Avg_Score', 'Max_Score', 'Unique_Docs']
    
    for idx, row in query_performance.iterrows():
        print(f"\nQuery {idx + 1}: {row['Matches']} matches, Avg: {row['Avg_Score']:.3f}")
        print(f"  '{relevant_sentences[idx][:80]}...'")
        print(f"  Unique documents: {row['Unique_Docs']}, Max score: {row['Max_Score']:.3f}")
    
    # Document distribution
    print(f"\n📚 DOCUMENT DISTRIBUTION:")
    doc_counts = results_df['document_id'].value_counts().head(10)
    print(f"Top 10 documents with most relevant sentences:")
    for doc_id, count in doc_counts.items():
        avg_score = results_df[results_df['document_id'] == doc_id]['similarity_score'].mean()
        print(f"  Document {doc_id}: {count} sentences (avg score: {avg_score:.3f})")
    
    # Score distribution
    print(f"\n📈 SCORE DISTRIBUTION:")
    score_ranges = [
        (0.9, 1.0, "Excellent (0.9-1.0)"),
        (0.8, 0.9, "Very Good (0.8-0.9)"),
        (0.7, 0.8, "Good (0.7-0.8)"),
        (0.6, 0.7, "Moderate (0.6-0.7)"),
        (0.0, 0.6, "Lower (0.0-0.6)")
    ]
    
    for min_score, max_score, label in score_ranges:
        count = len(results_df[(results_df['similarity_score'] >= min_score) & 
                              (results_df['similarity_score'] < max_score)])
        percentage = count / len(results_df) * 100
        print(f"  {label}: {count} sentences ({percentage:.1f}%)")

In [9]:
def show_top_cross_encoder_results(results_df, top_n=10):
    """
    Display top N results in a readable format
    """
    
    print(f"\n" + "="*80)
    print(f"TOP {top_n} CROSS-ENCODER RESULTS")
    print("="*80)
    
    for idx, row in results_df.head(top_n).iterrows():
        print(f"\n🏆 RANK {idx + 1}")
        print(f"📄 Document ID: {row['document_id']}")
        print(f"🎯 Similarity Score: {row['similarity_score']:.4f}")
        print(f"🔍 Matched Query: {row['query_text'][:100]}...")
        print(f"📝 Sentence: {row['sentence_text']}")
        print(f"🆔 Sentence ID: {row['sentence_id']}")
        print("-" * 80)

# Additional function to test different cross-encoder models
def compare_cross_encoder_models(relevant_sentences, document_sentences_df, 
                                sample_size=100, threshold=0.5):
    """
    Compare different cross-encoder models on a sample of your data
    """
    
    models_to_test = [
        'cross-encoder/ms-marco-MiniLM-L-6-v2',      # Fast, general purpose
        'cross-encoder/ms-marco-MiniLM-L-12-v2',     # Better quality
        'cross-encoder/stsb-roberta-large',          # High quality for semantic similarity
        'cross-encoder/nli-deberta-v3-large'         # Very high quality
    ]
    
    # Use sample of data for comparison
    sample_df = document_sentences_df.head(sample_size)
    
    results_comparison = {}
    
    for model_name in models_to_test:
        print(f"\n🧪 Testing model: {model_name}")
        
        start_time = time.time()
        
        results = cross_encoder_similarity_search(
            relevant_sentences, 
            sample_df,
            threshold=threshold,
            model_name=model_name,
            batch_size=16
        )
        
        elapsed_time = time.time() - start_time
        
        results_comparison[model_name] = {
            'matches_found': len(results),
            'avg_similarity': results['similarity_score'].mean() if len(results) > 0 else 0,
            'max_similarity': results['similarity_score'].max() if len(results) > 0 else 0,
            'processing_time_minutes': elapsed_time / 60,
            'unique_documents': results['document_id'].nunique() if len(results) > 0 else 0
        }
    
    # Display comparison
    comparison_df = pd.DataFrame(results_comparison).T
    print(f"\n📊 MODEL COMPARISON RESULTS (sample size: {sample_size}):")
    print(comparison_df.round(3))
    
    return comparison_df

In [10]:
# Your labeled relevant sentences (queries)
relevant_sentences = [
    "Land use and land cover changes impact biodiversity",
    "Urban expansion affects agricultural land",
    "LULC transitions influence ecosystem services",
    "Deforestation patterns show significant spatial variation"
]

# Test with a small sample first
print("🧪 TESTING WITH SMALL SAMPLE (first 500 sentences)")
test_results = cross_encoder_similarity_search(
    relevant_sentences=relevant_sentences,
    document_sentences_df=document_sentences_df,
    threshold=0.6,  # Start with lower threshold for cross-encoder
    batch_size=32,
    max_doc_sentences=500  # Test with first 500 sentences
)

# Analyze test results
if len(test_results) > 0:
    analyze_cross_encoder_results(test_results, relevant_sentences)
    show_top_cross_encoder_results(test_results, top_n=10)
    
    # Save test results
    test_results.to_csv('lulc_cross_encoder_test_results.csv', index=False)
    print(f"\nTest results saved to 'lulc_cross_encoder_test_results.csv'")

# If test results look good, run on full dataset
print(f"\n🚀 Ready to run on full dataset? ({len(document_sentences_df)} sentences)")
print("This might take 30-60 minutes depending on your dataset size...")

# Uncomment to run on full dataset:
# full_results = cross_encoder_similarity_search(
#     relevant_sentences=relevant_sentences,
#     document_sentences_df=document_sentences_df,
#     threshold=0.6,
#     batch_size=32
# )

🧪 TESTING WITH SMALL SAMPLE (first 500 sentences)
Loading Cross-Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2


NameError: name 'CrossEncoder' is not defined

In [14]:
print("\n" + "="*80)
print("TOP 10 MOST RELEVANT SENTENCES:")
print("="*80)

for idx, row in results.head(10).iterrows():
    print(f"\nRank: {idx+1}")
    print(f"Document ID: {row['document_id']}")
    print(f"Sentence Number: {row['sentence_number']}")
    print(f"Similarity Score: {row['similarity_score']:.3f}")
    print(f"Sentence: {row['sentence_text']}")
    print(f"Matched Query: {row['matching_query'][:1000]}")
    print("-" * 50)


TOP 10 MOST RELEVANT SENTENCES:

Rank: 43
Document ID: Article_9
Sentence Number: 40
Similarity Score: 0.940
Sentence: Generally, agriculture and built up areas are increasing at the expense of forest, wetland and woodland.
Matched Query: Thus farmland persists and increases each year at the expense of forest woodland and tree savanna . 
--------------------------------------------------

Rank: 132
Document ID: Article_19
Sentence Number: 67
Similarity Score: 0.934
Sentence: The Atlantic Forest and Pampa biomes also showed a minor trend towards an increase in surface water and higher variation over the time-series period.
Matched Query: In contrast relative net gain in open water bodies was observed in the humid region during the period of 1975 2000 with a slight loss in 2000 2013. 
--------------------------------------------------

Rank: 500
Document ID: Article_63
Sentence Number: 5
Similarity Score: 0.931
Sentence: Over the last 33 years, cultivation and settlement land expanded b

In [ ]:
results.to_csv('lulc_relevant_sentences_results.csv', index=False)
print(f"\nResults saved to 'lulc_relevant_sentences_results.csv'")

# ================================
# STEP 7: EVALUATION METRICS
# ================================

print(f"\nSummary Statistics:")
print(f"Total sentences processed: {len(document_sentences_df)}")
print(f"Relevant sentences found: {len(results)}")
print(f"Coverage: {len(results)/len(document_sentences_df)*100:.2f}% of sentences")
print(f"Average similarity score: {results['similarity_score'].mean():.3f}")
print(f"Min similarity score: {results['similarity_score'].min():.3f}")
print(f"Max similarity score: {results['similarity_score'].max():.3f}")

# Distribution by document
doc_counts = results['document_id'].value_counts()
print(f"\nTop 5 documents with most relevant sentences:")
print(doc_counts.head())